### Diffuse Irradiation with PVlib


install [pvlib package](https://pvlib-python.readthedocs.io/en/stable/installation.html)


In [3]:
import sys
import os
import getpass
import pandas as pd
import numpy as np
import json
from sqlalchemy import *

# notebook
from IPython.display import Image
from IPython.core.display import HTML 

# pvlib
import pvlib


version = 'v1 (jupyter)'
project = 'pv3'

In [4]:
# check for csv files in upper directory
for root, dirs, files in os.walk('./'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

./data\2020-05-12_pv3_sonnja_plot_all_wr.csv
./data\2020-06-10_pv3_sonnja_plot_all_wr.csv
./tutorial\vis-tutorial-master\data\conventional_power_plants_DE.csv
./tutorial\vis-tutorial-master\data\conventional_power_plants_EU.csv
./tutorial\vis-tutorial-master\data\time_series_60min_singleindex.csv


### values needed in polysun
- [x] Gh - Globalstrahlung [Wh/m2]
- [ ] Dh  - Diffusstrahlung [Wh/m2]
- [ ] Lh - Langwellenstrahlung [Wh/m2]
- [x] Tamb Umgebungstemperatur [°C]
- [x] Vwnd Wind [m/s]
- [x] Hrel Luftfeuchtigkeit [%]

## import htw weatherdata

In [6]:
df_htw = pd.read_csv('D:\git\github\htw-pv3\pvlib-python-pv3\data\pv3_2015\htw_wetter_weatherdata_2015.csv', encoding='latin1', sep=';', index_col=0, parse_dates=True)#, skiprows=3)
df_htw.head()

,G_hor_CMP6,G_hor_Si,G_gen_CMP11,G_gen_Si,Ev_Beleuchtung,v_Wind,d_Wind,T_Luft,h_Luft,p_Luft,i_Niederschlag
timestamp,,,,,,,,,,,
2015-01-01 00:00:00,0,0.0,0.0,3,0.0,0.0,0,0.0,0.0,0.0,0.0
2015-01-01 00:01:00,0,0.0,0.0,3,0.0,0.0,0,0.0,0.0,0.0,0.0
2015-01-01 00:02:00,0,0.0,0.0,3,0.0,0.0,0,0.0,0.0,0.0,0.0
2015-01-01 00:03:00,0,0.0,0.0,3,0.0,0.0,0,0.0,0.0,0.0,0.0
2015-01-01 00:04:00,0,0.0,0.0,2,0.0,0.0,0,0.0,0.0,0.0,0.0


### resample

In [7]:
df_htw_h = df_htw.resample('H').mean()
df_htw_h.head()

,G_hor_CMP6,G_hor_Si,G_gen_CMP11,G_gen_Si,Ev_Beleuchtung,v_Wind,d_Wind,T_Luft,h_Luft,p_Luft,i_Niederschlag
timestamp,,,,,,,,,,,
2015-01-01 00:00:00,0.0,0.0,0.0,2.483333,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2015-01-01 01:00:00,0.0,0.0,0.0,2.400000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2015-01-01 02:00:00,0.0,0.0,0.0,2.472727,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2015-01-01 03:00:00,0.0,0.0,0.0,2.450000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2015-01-01 04:00:00,0.0,0.0,0.0,2.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
fig = px.line(df_htw_h, title='htw_wetter_weatherdata_2015')

# deactivate all but first column
for i, _ in enumerate(fig.data):
    if i==0: 
        fig.data[i].update({'visible':True})
    else:
        fig.data[i].update({'visible':'legendonly'})
fig.show()

NameError: name 'px' is not defined

In [9]:
# HTW coords
lat = 52.455778
lon = 13.523917

### compute solarpositiont / zenith

In [10]:
df_solarpos = pvlib.solarposition.spa_python(df_htw_h.index, lat, lon)
df_solarpos.head()

,apparent_zenith,zenith,apparent_elevation,elevation,azimuth,equation_of_time
timestamp,,,,,,
2015-01-01 00:00:00,149.016792,149.016792,-59.016792,-59.016792,23.190805,-3.185267
2015-01-01 01:00:00,143.775044,143.775044,-53.775044,-53.775044,46.415709,-3.204993
2015-01-01 02:00:00,136.242961,136.242961,-46.242961,-46.242961,64.508060,-3.224709
2015-01-01 03:00:00,127.574379,127.574379,-37.574379,-37.574379,78.989205,-3.244416
2015-01-01 04:00:00,118.482773,118.482773,-28.482773,-28.482773,91.398984,-3.264114


In [11]:
df_irradiance = pvlib.irradiance.erbs(ghi=df_htw_h.loc[:,'G_hor_CMP6'],
                      zenith=df_solarpos.zenith,
                      datetime_or_doy=df_htw_h.index.dayofyear)
df_irradiance = pd.DataFrame(df_irradiance)

In [12]:
df_irradiance

,dni,dhi,kt
timestamp,,,
2015-01-01 00:00:00,0.0,0.000000,0.000000
2015-01-01 01:00:00,0.0,0.000000,0.000000
2015-01-01 02:00:00,0.0,0.000000,0.000000
2015-01-01 03:00:00,0.0,0.000000,0.000000
2015-01-01 04:00:00,0.0,0.000000,0.000000
...,...,...,...
2015-12-31 19:00:00,0.0,2.533333,0.027564
2015-12-31 20:00:00,0.0,3.350000,0.036450
2015-12-31 21:00:00,0.0,3.900000,0.042435


## Dataformat import